#### **Library Imports**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from gtm_boardroom.data.generator import GTM_DataGenerator
from gtm_boardroom.data.schemas import DataConfig, OEMTierConfig

# Set visual style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [12, 6]

#### **Data Generation for Rank 3 Competitor [The Upstart]**

In [ ]:
# Initialize Generator for Rank 3 (Upstart)
from gtm_boardroom.data.config import get_tier_config

data_cfg, oem_cfg, coeffs = get_tier_config('upstart')
promos = []

generator = GTM_DataGenerator(data_cfg, oem_cfg, coeffs, promos=promos)
df = generator.generate()

print(f"Generated {len(df)} weeks of market data for Rank 3 OEM.")

#### **The Marketing Physics (The Hill Function)**

In a saturated market, an Upstart's biggest enemy isn't the competitor, it's the Noise Floor. The physics engine uses a Hill Function with n=2.2. For Rank 3 brands, this means marketing spend under a certain threshold are effectively invisible. There must be a minimum spend to 'break through' before even a single dollar of ROI is seen

In [ ]:
def hill_function(spend, k, n):
    return (spend**n) / (spend**n + k**n)

spend_range = np.linspace(0, 200000, 500)

# Physics of the Leader vs Upstart
leader_roi = hill_function(spend_range, 30000, 1.1)
challenger_roi = hill_function(spend_range, 55000, 1.5)
upstart_roi = hill_function(spend_range, 80000, 2.2)

plt.figure(figsize=(10, 6))
plt.plot(spend_range, leader_roi, label='Rank 1 (Leader): Efficient, Immediate Impact', lw=3)
plt.plot(spend_range, challenger_roi, label='Rank 2 (Challenger): Balanced Approach', lw=3, color='orange')
plt.plot(spend_range, upstart_roi, label='Rank 3 (Upstart): S-Curve, High Barrier', lw=3, color='red')
plt.axvline(x=80000, color='gray', linestyle='--', label='Upstart Saturation Point (K)')

plt.title("The 'Noise Floor' Phenomenon: Marketing ROI by OEM Rank")
plt.xlabel("Marketing Spend ($)")
plt.ylabel("ROI Multiplier (0 to 1)")
plt.legend()
plt.show()

#### **Price Elasticity & Competitive Pressure**

Price isn't an absolute value; it's a relative one. For an Upstart, being $50 more expensive than a Market Leader isn't a premium, it's a sales death sentence. The data generator models this 'Causal Penalty,' showing the narrow green corridor where a Rank 3 brand can actually survive.

In [ ]:
# Create a heatmap of Sales based on Our Price vs Leader Price
prices = np.linspace(300, 800, 20)
leader_prices = np.linspace(300, 800, 20)
price_grid = np.zeros((20, 20))

# Fixed baseline for visualization
base = 20000 # Baseline for Rank 3
base_price = 799 

for i, p in enumerate(prices):
    for j, lp in enumerate(leader_prices):
        # 1. Price Elasticity Effect
        price_effect = np.exp(1.8 * (1 - (p/base_price)))
        # 2. Competitive Penalty (If our price > Leader * 1.05)
        penalty = 0.9 if p > (lp * 1.05) else 1.0
        price_grid[i, j] = base * price_effect * penalty

plt.figure(figsize=(10, 8))
sns.heatmap(price_grid, xticklabels=np.round(leader_prices,0), yticklabels=np.round(prices,0), cmap="RdYlGn")
plt.title("The 'Red Ocean' Trap: Sales as a function of Our Price vs Leader Price")
plt.xlabel("Market Leader Price ($)")
plt.ylabel("Upstart Price ($)")
plt.show()

####  **Time-Series Decomposition (The "Anatomy of a Week")**

Decomposing a high-performing week to understand what caused the spike is fundamental to breaking down its performance


In [ ]:
# Select a subset of data for clarity
#plot_df = df.iloc[50:70].copy()
plot_df = df.iloc[0:155].copy()

fig, ax1 = plt.subplots(figsize=(14, 7))

# Plot Sales
ax1.plot(plot_df['week'], plot_df['sales'], label='Weekly Sales', color='black', lw=2)
ax1.set_ylabel('Unit Sales')

# Overlay Marketing & Launch Spikes
ax2 = ax1.twinx()
ax2.fill_between(plot_df['week'], 0, plot_df['search_spend'], alpha=0.2, color='blue', label='Marketing Spend')
ax2.plot(plot_df['week'], plot_df['oem_launch_spike'] * 10000, color='orange', linestyle='--', label='Launch Window [Flagship]')
ax2.set_ylabel('Spend / Events')

plt.title("Anatomy of GTM: How Launch Spikes and Marketing ROI converge")
fig.legend(loc="upper right", bbox_to_anchor=(1,1), bbox_transform=ax1.transAxes)
plt.show()

#### **The "Marketing Memory" (Adstock Decay)**

Marketing doesn't vanish the moment a campaign ends. The Data Generator class implements an Exponential Weighted Moving Average (EWMA) decay to simulate 'Adstock.' It allows the data (digital twin) to mimic the behavior that sustained brand presence builds more momentum than isolated spikes; a critical factor for Rank 3 brands trying to build trust.

In [ ]:
# Compare Raw Spend vs Adstock (Marketing Memory)
plt.figure(figsize=(14, 6))

# Zoom into 20 weeks for clarity
sample_df = df.iloc[0:50]

plt.bar(sample_df['week'], sample_df['search_spend'], alpha=0.3, label='Raw Search Spend (Input)', color='gray')
plt.plot(sample_df['week'], sample_df['search_adstock'], marker='o', label='Search Adstock (Market Memory)', color='blue', lw=2)

plt.title("Physics of 'Market Memory': Proving Adstock Decay Logic")
plt.ylabel("Effective Marketing Pressure ($)")
plt.legend()
plt.show()

#### **The Revenue vs. Volume "Sweet Spot"**

Data Science isn't just about predicting sales; it's about optimizing value. Revenue Peaks let's us understand that while $550 and $650 genrate similar revenue, however the volume takes a hit with price increase as Elasticity takes over. For an upstart brand fighting for market share this makes a big difference.

In [ ]:
# Calculate Revenue
df['revenue'] = df['sales'] * df['list_price']

plt.figure(figsize=(12, 6))
# Binning price into $50 buckets for a cleaner view
df['price_bucket'] = (df['list_price'] // 50) * 50

# Grouping by price bucket to find average revenue
rev_opt = df.groupby('price_bucket')[['sales', 'revenue']].mean().reset_index()

fig, ax1 = plt.subplots()

ax2 = ax1.twinx()
ax1.bar(rev_opt['price_bucket'], rev_opt['sales'], alpha=0.4, color='green', label='Avg Weekly Units', width=40)
ax2.plot(rev_opt['price_bucket'], rev_opt['revenue'], marker='s', color='darkred', label='Avg Weekly Revenue', lw=3)

ax1.set_xlabel('Price Point ($)')
ax1.set_ylabel('Units Sold', color='green')
ax2.set_ylabel('Total Revenue ($)', color='darkred')
plt.title("The GTM Sweet Spot: Finding the Revenue-Maximizing Price Point")
plt.show()

#### **The Marginal Breakthrough Point (Derivative Analysis)**

For an Upstart, spend is not linear. Using the first derivative of my Hill Function, I've identified the Breakthrough Point (roughly $51k in this model). This is the 'Inflection Point' where brand awareness overcomes market noise. Spending below this point is wasted; spending above it is where the Rank 3 brand starts to see returns or gain real market share.

In [ ]:
# Calculating the Marginal Return of Marketing (Slope of the Hill Function)
spend_range = np.linspace(1, 150000, 1000)
k, n = 80000, 2.2 # Rank 3 Params
roi = (spend_range**n) / (spend_range**n + k**n)

# Derivative of Hill Function (Marginal ROI)
marginal_roi = np.gradient(roi, spend_range)

plt.figure(figsize=(12, 6))
plt.plot(spend_range, marginal_roi * 100000, label='Marginal ROI (Slope)', color='purple', lw=3)
plt.axvline(x=spend_range[np.argmax(marginal_roi)], color='red', linestyle='--', label='Maximum Efficiency Point')

plt.title("Efficiency Analysis: Identifying the 'Breakthrough' Threshold")
plt.xlabel("Marketing Spend ($)")
plt.ylabel("Marginal Impact per Dollar")
plt.legend()
plt.show()